# 🏭 Visualisation Process Industriel — VS Code compatible

> **Fonctionne sous VS Code sans droits admin.**  
> Le graphe s'ouvre dans votre **navigateur** (Chrome/Edge) — zoomable, déplaçable, tooltips riches.

**Installation (une seule fois) :**
```
pip install pyvis pandas
```

In [122]:
# !pip install pyvis pandas

import json, webbrowser
import pandas as pd
from pathlib import Path
from pyvis.network import Network
from IPython.display import display, HTML

print('✅ OK')

✅ OK


## 1. Chargement des données

In [123]:
BASE = Path('.')

def jload(f):
    with open(BASE / f, encoding='utf-8') as fh:
        return json.load(fh)

sites = jload('sites.json')['sites']
mps   = jload('matieres_premieres.json')['matieres_premieres']
ups   = jload('unites_production.json')['unites_production']

site_map = {s['code']: s for s in sites}
mp_map   = {m['code']: m for m in mps}
up_map   = {u['code']: u for u in ups}

print(f'✅ {len(sites)} sites | {len(mps)} MP | {len(ups)} UP')

✅ 4 sites | 10 MP | 11 UP


## 2. Helpers couleurs

In [124]:
def oee_color(v):
    return '#1E8449' if v >= 85 else '#D35400' if v >= 75 else '#C0392B'

def oee_icon(v):
    return '🟢' if v >= 85 else '🟡' if v >= 75 else '🔴'

def pct_stock(mp):
    try:    return round(mp['volume_stock'] / mp['stock_max'] * 100)
    except: return 0

def stock_color(mp):
    return '#C0392B' if mp['volume_stock'] <= mp['stock_min'] else \
           '#D35400' if pct_stock(mp) < 30 else '#1E8449'

def mini_bar(val, width=120):
    """Barre de progression en HTML pour les tooltips pyvis."""
    fill = int(val / 100 * width)
    color = oee_color(val)
    return (
        f'<div style="background:#eee;width:{width}px;height:8px;border-radius:4px;display:inline-block">'
        f'<div style="background:{color};width:{fill}px;height:8px;border-radius:4px"></div>'
        f'</div>'
    )

NODE_COLORS = {
    'matiere_premiere'     : {'bg': '#C0392B', 'border': '#922B21'},
    'additif'              : {'bg': '#C0392B', 'border': '#922B21'},
    'consommable'          : {'bg': '#E74C3C', 'border': '#922B21'},
    'emballage'            : {'bg': '#8E44AD', 'border': '#6C3483'},
    'produit_intermediaire': {'bg': '#E67E22', 'border': '#A04000'},
    'produit_fini'         : {'bg': '#2980B9', 'border': '#1A5276'},
}

print('✅ Helpers OK')

✅ Helpers OK


## 3. Construction du graphe PyVis

In [125]:
def build_graph(
    show_mp       = True,
    site_filter   = None,
    output_file   = 'process_industriel.html',
    hierarchical  = True,
    direction     = 'LR',        # 'LR' gauche→droite | 'UD' haut→bas
):
    net = Network(
        height        = '820px',
        width         = '100%',
        directed      = True,
        bgcolor       = '#F4F6F7',
        font_color    = '#2C3E50',
        notebook      = False,
    )

    # Layout hiérarchique propre (comme graphviz dot)
    if hierarchical:
        dir_map = {'LR': 'RL', 'UD': 'DU', 'TB': 'DU'}
        net.set_options(f"""{{
          "layout": {{
            "hierarchical": {{
              "enabled": true,
              "direction": "{direction}",
              "sortMethod": "directed",
              "nodeSpacing": 180,
              "levelSeparation": 260,
              "treeSpacing": 200
            }}
          }},
          "physics": {{"enabled": false}},
          "interaction": {{
            "hover": true,
            "tooltipDelay": 100,
            "navigationButtons": true,
            "keyboard": true
          }},
          "edges": {{
            "smooth": {{"type": "cubicBezier", "forceDirection": "{direction}"}},
            "arrows": {{"to": {{"enabled": true, "scaleFactor": 1.2}}}}
          }}
        }}""")

    ups_f    = [u for u in ups if site_filter is None or u['site_code'] == site_filter]
    up_codes = {u['code'] for u in ups_f}

    # ── Nœuds Matières premières ──────────────────────────────────────────────
    if show_mp:
        for mp in mps:
            if site_filter and mp['site_code'] != site_filter:
                continue
            p    = pct_stock(mp)
            sc   = stock_color(mp)
            icon = '🔴' if mp['volume_stock'] <= mp['stock_min'] else '🟡' if p < 30 else '🟢'
            colors = NODE_COLORS.get(mp['type'], {'bg':'#C0392B','border':'#922B21'})

            label   = f"{mp['nom']}\n{mp['volume_stock']:,} {mp['unite_volume']}"
            tooltip = (
                f"<div style='font-family:Arial;font-size:13px;padding:10px;min-width:200px'>"
                f"<b style='color:{colors['bg']};font-size:15px'>{mp['nom']}</b><br>"
                f"<hr style='margin:5px 0'>"
                f"<b>Code :</b> {mp['code']}<br>"
                f"<b>Type :</b> {mp['type']}<br>"
                f"<b>Site :</b> {site_map.get(mp['site_code'],{}).get('nom','')}<br>"
                f"<b>Fournisseur :</b> {mp.get('fournisseur','-')}<br>"
                f"<hr style='margin:5px 0'>"
                f"<b>Stock :</b> <span style='color:{sc}'>{icon} {mp['volume_stock']:,} {mp['unite_volume']}</span><br>"
                f"Stock min : {mp['stock_min']:,} | max : {mp['stock_max']:,}<br>"
                f"{mini_bar(p)} {p}% du max<br>"
                f"<b>Délai appro :</b> {mp.get('delai_approvisionnement_j','-')} j"
                f"</div>"
            )
            net.add_node(
                mp['code'],
                label  = label,
                title  = tooltip,
                shape  = 'diamond',
                color  = {'background': colors['bg'],
                          'border'    : colors['border'],
                          'highlight' : {'background': '#F9E79F', 'border': '#D4AC0D'}},
                size   = 28,
                font   = {'color': 'white', 'size': 11, 'bold': True},
                group  = 'mp',
            )

    # ── Nœuds Unités de production ────────────────────────────────────────────
    for u in ups_f:
        oc    = oee_color(u['oee'])
        taux  = round(u['cmj'] / u['capacite_max_j'] * 100)
        site  = site_map.get(u['site_code'], {}).get('nom', u['site_code'])
        scol  = site_map.get(u['site_code'], {}).get('couleur', '#2C3E50')
        colors= NODE_COLORS.get(u['type_produit'], {'bg':'#2980B9','border':'#1A5276'})
        main  = u['type_ligne'] == 'principale'

        label = (
            f"{u['nom']}\n"
            f"{oee_icon(u['oee'])} OEE {u['oee']}%  |  CMJ {u['cmj']:,}"
        )
        tooltip = (
            f"<div style='font-family:Arial;font-size:13px;padding:10px;min-width:240px'>"
            f"<b style='color:{colors['bg']};font-size:15px'>{u['nom']}</b><br>"
            f"<span style='color:{scol}'>{site}</span>"
            f" &nbsp;•&nbsp; {u['type_ligne'].upper()}<br>"
            f"<hr style='margin:5px 0'>"
            f"📦 <b>Produit :</b> {u['produit']}<br>"
            f"<b>Code :</b> {u['code']}<br>"
            f"<hr style='margin:5px 0'>"
            f"<b>OEE :</b> <span style='color:{oc};font-size:15px'><b>{oee_icon(u['oee'])} {u['oee']}%</b></span><br>"
            f"{mini_bar(u['oee'])} &nbsp;"
            f"{'✅ Objectif atteint' if u['oee']>=85 else '⚠️ Sous objectif (cible 85%)'}<br>"
            f"<hr style='margin:5px 0'>"
            f"<b>CMJ :</b> {u['cmj']:,} {u['unite_capacite']}<br>"
            f"<b>Cap. max/j :</b> {u['capacite_max_j']:,}<br>"
            f"<b>Taux de charge :</b> {taux}%<br>"
            f"{mini_bar(taux, width=120)}<br>"
            f"<b>Notes :</b> {u.get('notes','—')}"
            f"</div>"
        )
        net.add_node(
            u['code'],
            label    = label,
            title    = tooltip,
            shape    = 'box' if u['type_produit'] == 'produit_fini' else 'ellipse',
            color    = {'background': colors['bg'],
                        'border'    : scol,          # bordure = couleur du site
                        'highlight' : {'background': '#F9E79F', 'border': '#D4AC0D'}},
            size     = 40 if main else 28,
            borderWidth = 4 if main else 1,
            font     = {'color': 'white', 'size': 13 if main else 11, 'bold': main},
            group    = u['site_code'],
        )

    # ── Arêtes MP → UP ────────────────────────────────────────────────────────
    if show_mp:
        for u in ups_f:
            for src in u.get('matieres_premieres_aval', []):
                if src.startswith('MP-') and src in mp_map:
                    if site_filter and mp_map[src]['site_code'] != site_filter:
                        continue
                    net.add_edge(
                        src, u['code'],
                        color  = '#E74C3C',
                        dashes = True,
                        width  = 1.5,
                        title  = f"Approvisionnement : {mp_map[src]['nom']}",
                        label  = mp_map[src]['nom'][:16],
                        font   = {'size': 9, 'color': '#C0392B', 'align': 'middle'},
                    )

    # ── Arêtes UP → UP ────────────────────────────────────────────────────────
    for u in ups_f:
        for nxt in u.get('produits_suivants', []):
            if nxt not in {x['code'] for x in ups}:
                continue
            if site_filter and nxt not in up_codes:
                continue
            main = u['type_ligne'] == 'principale'
            net.add_edge(
                u['code'], nxt,
                color  = '#2C3E50' if main else '#95A5A6',
                width  = 4 if main else 1.5,
                dashes = not main,
                title  = f"{'Flux principal' if main else 'Flux secondaire'} : {u['produit']}",
                label  = u['produit'][:20] if main else '',
                font   = {'size': 9, 'color': '#2C3E50', 'align': 'middle'},
            )

    # Sauvegarde + ouverture navigateur
    out = Path(output_file)
    net.save_graph(str(out))
    webbrowser.open(out.resolve().as_uri())
    print(f'✅ Graphe ouvert dans le navigateur → {out.resolve()}')
    return net

print('✅ build_graph() prête')

✅ build_graph() prête


## 4. 🗺️ Vue complète — tous sites
*Le graphe s'ouvre dans votre navigateur. Molette pour zoomer, clic-glisser pour déplacer, survol pour les détails.*

In [126]:
build_graph(show_mp=True, direction='LR', output_file='vue_complete.html')

✅ Graphe ouvert dans le navigateur → C:\Users\eplaidy\OneDrive - ADISSEO\Documents\GitHub\Machine-Learning\1-Projet\Industriels\vue_complete.html


<class 'pyvis.network.Network'> |N|=21 |E|=20

## 5. ⚙️ Flux principaux — sans matières premières

In [127]:
build_graph(show_mp=False, direction='UD', output_file='vue_flux.html')

✅ Graphe ouvert dans le navigateur → C:\Users\eplaidy\OneDrive - ADISSEO\Documents\GitHub\Machine-Learning\1-Projet\Industriels\vue_flux.html


<class 'pyvis.network.Network'> |N|=11 |E|=10

## 6. 🔍 Vue par site

In [128]:
for s in sites:
    print(f"  {s['code']}  →  {s['nom']} ({s['localisation']})")

# ← Changer le site ici
build_graph(show_mp=True, site_filter='ROC',
            direction='UD', output_file='vue_site_A.html')

  ROC  →  Les Roches de Condrieu (Lyon)
  BGS  →  Usine de Burgos (Burgos)
  BAY  →  Fondoir de Bayonne (Bayonne)
  RON  →  Roussillon (Roussillon)
✅ Graphe ouvert dans le navigateur → C:\Users\eplaidy\OneDrive - ADISSEO\Documents\GitHub\Machine-Learning\1-Projet\Industriels\vue_site_A.html


<class 'pyvis.network.Network'> |N|=10 |E|=10

---
## 💡 Référence rapide

| Action | Code |
|---|---|
| Vue complète | `build_graph()` |
| Sans MP | `build_graph(show_mp=False)` |
| Haut→bas | `build_graph(direction='UD')` |
| Filtrer un site | `build_graph(site_filter='SITE-B')` |
| Fichier de sortie | `build_graph(output_file='mon_process.html')` |
| Ajouter un nœud | Éditer le JSON correspondant |